# 0️⃣3️⃣ - How to structure and manage scripts
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to structure scripts for reuse, and correctly manage the `Client` lifecycle in scripts, Jupyter Notebooks, and FastAPI apps.

### 🛠️ Before You Begin

If you haven't done it yet, follow the instructions [in this file](../SETUP.md) to setup your environment and run this playbook.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | Use a generator function to share a single `Client` across multiple functions in a script |
| 2 | Persist a `Client` instance across Jupyter Notebook cells with `ExitStack` |
| 3 | Structure a web application (FastAPI) around a single authenticated `Client` |

---

### 🔁 Context Manager (default behaviour)

As you have seen already in the [previous notebooks](01-authentication-connection.ipynb), the creation of a `Client` is always surrounded by a Context Manager: `with Client.create() as client: ...`. This is intended to be a guardrail for the SDK to handle by itself the creation and destruction of the instance.

This can be problematic when you need persistance of the same `Client` instance across the whole script through different functions, or when you are creating something that is not a script per se. In the following sections, we will learn how to tweak this a little bit to adjust to your projects.

---

### 📦 Yielding your `Client`

A generator function lets you create **one authenticated `Client` instance** and share it freely across your entire script without repeating the `with` block.

**How it works:**
1. A generator function wraps `Client.create()` and `sso_login()`, then **yields** the ready client.
2. Calling `next()` on the generator runs everything up to `yield`, returning the client.
3. When you're done, calling `.close()` on the generator resumes the function past `yield`, which exits the `with` block and cleans up the client properly.

In [2]:
from radkit_client import Client


def managed_client(user_id: str):
    """Generator that yields a single authenticated Client instance."""
    with Client.create() as client:
        client.sso_login(user_id)
        yield client  # execution pauses here; client stays alive


def task_one(client: Client):
    print(f"[Task 1️⃣] Client status: {client.status}")


def task_two(client: Client):
    print(f"[Task 2️⃣] Client status: {client.status}")


# main script
user_id = input("👤 Enter your CCO user id: ")

_session = managed_client(user_id)
client = next(_session)  # authenticates + one instance created

task_one(client)
task_two(client)

_session.close()  # triggers cleanup — exits the `with Client.create()` block



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
[Task 1️⃣] Client status: ClientStatus.CONNECTED
[Task 2️⃣] Client status: ClientStatus.CONNECTED


---

### 📙 Jupyter Notebooks

It is also possible to persist your single `Client` instance through the different cells of a notebook:

In [3]:
from contextlib import ExitStack
from radkit_client.sync import Client

stack = ExitStack()  # This is a Context Manager that allows us to manage the lifecycle of multiple resources (like our Client instance) in a clean and efficient way.
my_client = stack.enter_context(Client.create())  # We create a Client instance and register it

user_id = input("👤 Enter your CCO user id: ")
my_client.sso_login(user_id)


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.


Client(status='CONNECTED')

We can use the same `Client` across multiple cells:

In [ ]:
print(f"[Cell 1️⃣] Client status: {my_client.status}")

[Cell 1️⃣] Client status: ClientStatus.CONNECTED


In [ ]:
print(f"[Cell 2️⃣] Client status: {my_client.status}")

[Cell 2️⃣] Client status: ClientStatus.CONNECTED


Then at the end, we can simply close the stack to terminate the `Client` instance:

In [4]:
stack.close()

---

### 🌐 Web Applications

If you want to make your single `Client` instance available for multiple HTTP requests in a web app, there are a series of recommendations that you need to follow.

We have an example based on a **FastAPI web server** with several basic API endpoints available that make use of a single authenticated `Client` instance.

The example is available [in this repository](https://github.com/Cisco-RADKit/radkit-fastapi).

---

## 🔀 Which Method Should I Use?

| | 📦 Generator (Script) | 📙 ExitStack (Notebook) | 🌐 Web App (FastAPI) |
|---|---|---|---|
| **Best for** | CLI scripts, automation tasks | Interactive Jupyter sessions | Long-running HTTP servers |
| **Lifecycle scope** | Single script execution | Notebook kernel lifetime | Application lifetime |
| **Sharing the client** | Pass as argument to functions | Access variable across cells | Access global variable across API endpoints |
| **Cleanup** | `_session.close()` | `stack.close()` | App shutdown event handler |
| **Overhead** | Minimal | Minimal | Slightly higher (app wiring) |